## Prerequisites

**Run these first:**
1. ✅ `preprocessing_FIXED.ipynb`
2. ✅ `01_setup_and_config.ipynb`

In [ ]:
# Restore from setup
%store -r config
%store -r device

import sys
import os
sys.path.insert(0, os.getcwd())

from utils import (
    set_seed, get_dataloaders, validate_checkpoint_fresh,
    train_one_epoch, validate, evaluate_model,
    plot_confusion_matrix, plot_roc_curves
)
from models import get_model, count_parameters, print_model_summary

import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
import time

print("✓ Setup complete")

## 1. Define Training Function

In [ ]:
def train_baseline_model(model_name, model, train_loader, val_loader, test_loader,
                        config, device, max_epochs=50):
    """
    Train a baseline model with consistent protocol.
    
    Args:
        model_name: Name of the model
        model: PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        test_loader: Test data loader
        config: Configuration dictionary
        device: Device (CPU/GPU)
        max_epochs: Maximum training epochs
    
    Returns:
        Dictionary with results and training time
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*80}\n")
    
    start_time = time.time()
    
    # Setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['optimizer']['learning_rate'],
        weight_decay=config['optimizer']['weight_decay']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode=config['scheduler']['mode'],
        factor=config['scheduler']['factor'],
        patience=config['scheduler']['patience']
    )
    scaler = torch.cuda.amp.GradScaler() if config['training']['mixed_precision'] else None
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    # Training loop
    best_val_acc = 0.0
    patience = 0
    max_patience = 7
    
    for epoch in range(1, max_epochs + 1):
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer,
            device, scaler, max_grad_norm=1.0, epoch=epoch
        )
        
        # Validate
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )
        
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch [{epoch:2d}/{max_epochs}] "
              f"Train: {train_loss:.4f}/{train_acc:6.2f}% | "
              f"Val: {val_loss:.4f}/{val_acc:6.2f}%")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience = 0
            # Save best model state
            best_model_state = model.state_dict().copy()
        else:
            patience += 1
            if patience >= max_patience:
                print(f"Early stopping at epoch {epoch}")
                break
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Evaluate on test set
    results = evaluate_model(
        model, test_loader, device, config['data']['class_names']
    )
    
    training_time = time.time() - start_time
    
    return {
        'model_name': model_name,
        'test_accuracy': float(results['accuracy']),
        'best_val_accuracy': float(best_val_acc),
        'parameters': count_parameters(model),
        'training_time_sec': training_time,
        'epochs_trained': epoch,
        'history': history,
        'confusion_matrix': results['confusion_matrix'].tolist(),
        'classification_report': results['classification_report']
    }

## 2. Load Data (Once for All Models)

In [ ]:
# Load data once - will be reused for all models
train_loader, val_loader, test_loader, heldout_test_loader = get_dataloaders(
    config=config,
    preprocessed_dir=config['paths']['preprocessed_data']
)

class_names = config['data']['class_names']
num_classes = len(class_names)

print(f"✓ Data loaded")
print(f"  Classes: {class_names}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 3. Define Models to Compare

In [ ]:
# Models to compare (name, description, pretrained)
models_to_compare = [
    ('tumornet_lite', 'TumorNet-Lite (Ours)', False),
    ('resnet50', 'ResNet-50', False),
    ('efficientnet_b0', 'EfficientNet-B0', False),
    ('mobilenet_v2', 'MobileNet-V2', False),
    ('mobilenet_v3_small', 'MobileNet-V3-Small', False),
    ('dmfnet', 'DMFNet', False)
]

print("Models to compare:")
for i, (name, desc, _) in enumerate(models_to_compare, 1):
    print(f"  {i}. {desc}")

## 4. Train All Models

**⚠️ Warning:** This will take 3-5 hours depending on GPU.  
Each model trains for up to 50 epochs with early stopping.

In [ ]:
# Store results for all models
all_results = []

# Train each model
for model_key, model_desc, use_pretrained in tqdm(models_to_compare, desc="Training models"):
    print(f"\n\n{'#'*80}")
    print(f"# MODEL {len(all_results) + 1}/{len(models_to_compare)}: {model_desc}")
    print(f"{'#'*80}\n")
    
    # Set seed for fair comparison
    set_seed(config['reproducibility']['seed'], config['reproducibility']['deterministic'])
    
    # Create model
    model = get_model(model_key, num_classes=num_classes, pretrained=use_pretrained)
    model = model.to(device)
    
    # Print summary
    print_model_summary(model, model_desc)
    
    # Train and evaluate
    try:
        result = train_baseline_model(
            model_name=model_desc,
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            config=config,
            device=device,
            max_epochs=50
        )
        all_results.append(result)
        
        print(f"\n✓ {model_desc} complete!")
        print(f"  Test Accuracy: {result['test_accuracy']:.2f}%")
        print(f"  Parameters: {result['parameters']:,}")
        print(f"  Training Time: {result['training_time_sec']/60:.1f} min")
        
    except Exception as e:
        print(f"\n✗ Error training {model_desc}: {str(e)}")
        continue
    
    # Clear GPU memory
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n\n{'='*80}")
print("ALL MODELS TRAINED!")
print(f"{'='*80}\n")

## 5. Results Summary Table

In [ ]:
# Create results DataFrame
df_results = pd.DataFrame([
    {
        'Model': r['model_name'],
        'Test Acc (%)': r['test_accuracy'],
        'Val Acc (%)': r['best_val_accuracy'],
        'Parameters (M)': r['parameters'] / 1e6,
        'Training Time (min)': r['training_time_sec'] / 60,
        'Epochs': r['epochs_trained']
    }
    for r in all_results
])

# Sort by test accuracy
df_results = df_results.sort_values('Test Acc (%)', ascending=False).reset_index(drop=True)

# Add rank
df_results.insert(0, 'Rank', range(1, len(df_results) + 1))

print("\n" + "="*100)
print("BASELINE COMPARISON RESULTS")
print("="*100)
print(df_results.to_string(index=False))
print("="*100 + "\n")

# Highlight best model
best_model = df_results.iloc[0]['Model']
print(f"🏆 Best Model: {best_model} ({df_results.iloc[0]['Test Acc (%)']:.2f}%)")

## 6. Visualizations

### 6.1 Accuracy Comparison

In [ ]:
# Bar plot of test accuracies
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if 'TumorNet-Lite' in m else '#3498db' for m in df_results['Model']]
bars = ax.barh(df_results['Model'], df_results['Test Acc (%)'], color=colors)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, df_results['Test Acc (%)'])):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{val:.2f}%', va='center', fontweight='bold')

ax.set_xlabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison - Test Accuracy', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(config['paths']['results'], 'baseline_accuracy_comparison.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Accuracy comparison saved")

### 6.2 Parameters vs Accuracy

In [ ]:
# Scatter plot: Model size vs accuracy
fig, ax = plt.subplots(figsize=(10, 6))

for i, row in df_results.iterrows():
    color = '#2ecc71' if 'TumorNet-Lite' in row['Model'] else '#3498db'
    marker = 's' if 'TumorNet-Lite' in row['Model'] else 'o'
    size = 200 if 'TumorNet-Lite' in row['Model'] else 100
    
    ax.scatter(row['Parameters (M)'], row['Test Acc (%)'], 
              c=color, marker=marker, s=size, alpha=0.7, edgecolors='black')
    
    # Add labels
    ax.annotate(row['Model'], 
               (row['Parameters (M)'], row['Test Acc (%)']),
               xytext=(5, 5), textcoords='offset points',
               fontsize=9, alpha=0.8)

ax.set_xlabel('Parameters (Millions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Model Efficiency: Parameters vs Accuracy', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config['paths']['results'], 'baseline_params_vs_accuracy.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Parameters vs accuracy plot saved")

### 6.3 Training Curves Comparison

In [ ]:
# Plot training curves for all models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for result in all_results:
    history = result['history']
    epochs = range(1, len(history['train_acc']) + 1)
    
    label = result['model_name']
    alpha = 1.0 if 'TumorNet-Lite' in label else 0.6
    linewidth = 2.5 if 'TumorNet-Lite' in label else 1.5
    
    # Accuracy
    ax1.plot(epochs, history['val_acc'], label=label, alpha=alpha, linewidth=linewidth)
    
    # Loss
    ax2.plot(epochs, history['val_loss'], label=label, alpha=alpha, linewidth=linewidth)

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax1.set_title('Validation Accuracy Curves', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(alpha=0.3)

ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Loss', fontsize=12)
ax2.set_title('Validation Loss Curves', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config['paths']['results'], 'baseline_training_curves.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training curves comparison saved")

### 6.4 Efficiency Metrics

In [ ]:
# Calculate efficiency scores
df_results['Acc per M Params'] = df_results['Test Acc (%)'] / df_results['Parameters (M)']
df_results['Acc per Min'] = df_results['Test Acc (%)'] / df_results['Training Time (min)']

# Plot efficiency metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy per million parameters
colors = ['#2ecc71' if 'TumorNet-Lite' in m else '#3498db' for m in df_results['Model']]
ax1.barh(df_results['Model'], df_results['Acc per M Params'], color=colors)
ax1.set_xlabel('Accuracy per Million Parameters', fontsize=12)
ax1.set_title('Parameter Efficiency', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# Accuracy per training minute
ax2.barh(df_results['Model'], df_results['Acc per Min'], color=colors)
ax2.set_xlabel('Accuracy per Training Minute', fontsize=12)
ax2.set_title('Training Efficiency', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(config['paths']['results'], 'baseline_efficiency_metrics.png'),
            dpi=300, bbox_inches='tight')
plt.show()

print("✓ Efficiency metrics saved")

## 7. Statistical Analysis

In [ ]:
# Find TumorNet-Lite results
tumornet_result = [r for r in all_results if 'TumorNet-Lite' in r['model_name']][0]
tumornet_acc = tumornet_result['test_accuracy']

print("\n" + "="*80)
print("STATISTICAL ANALYSIS")
print("="*80 + "\n")

print(f"TumorNet-Lite (Ours): {tumornet_acc:.2f}%\n")

# Compare with each baseline
for result in all_results:
    if 'TumorNet-Lite' not in result['model_name']:
        baseline_acc = result['test_accuracy']
        diff = tumornet_acc - baseline_acc
        relative_improvement = (diff / baseline_acc) * 100
        
        symbol = "✓" if diff > 0 else "✗"
        print(f"{symbol} vs {result['model_name']:25s}: "
              f"{diff:+6.2f}% (relative: {relative_improvement:+6.2f}%)")

print("\n" + "="*80)

# Summary statistics
print("\nSUMMARY STATISTICS:")
print(f"  Mean Accuracy: {df_results['Test Acc (%)'].mean():.2f}%")
print(f"  Std Dev: {df_results['Test Acc (%)'].std():.2f}%")
print(f"  Best: {df_results['Test Acc (%)'].max():.2f}%")
print(f"  Worst: {df_results['Test Acc (%)'].min():.2f}%")
print(f"  Range: {df_results['Test Acc (%)'].max() - df_results['Test Acc (%)'].min():.2f}%")

## 8. Save Complete Results

In [ ]:
# Save detailed results to JSON
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_path = os.path.join(
    config['paths']['results'],
    f'baseline_comparison_{timestamp}.json'
)

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"✓ Detailed results saved to {results_path}")

# Save summary table to CSV
csv_path = os.path.join(
    config['paths']['results'],
    f'baseline_comparison_summary_{timestamp}.csv'
)
df_results.to_csv(csv_path, index=False)

print(f"✓ Summary table saved to {csv_path}")

## 9. Final Summary

In [ ]:
print("\n" + "="*80)
print("BASELINE COMPARISON COMPLETE")
print("="*80 + "\n")

print("📊 Results Summary:")
print(f"  • {len(all_results)} models trained and evaluated")
print(f"  • Best model: {df_results.iloc[0]['Model']} ({df_results.iloc[0]['Test Acc (%)']:.2f}%)")
print(f"  • Total training time: {df_results['Training Time (min)'].sum():.1f} minutes")

print("\n📈 Visualizations Generated:")
print("  ✓ baseline_accuracy_comparison.png")
print("  ✓ baseline_params_vs_accuracy.png")
print("  ✓ baseline_training_curves.png")
print("  ✓ baseline_efficiency_metrics.png")

print("\n💾 Files Saved:")
print(f"  ✓ {os.path.basename(results_path)}")
print(f"  ✓ {os.path.basename(csv_path)}")

print("\n🎯 Key Findings:")
# Automatically generate key findings
best_idx = df_results['Test Acc (%)'].idxmax()
most_efficient_idx = df_results['Acc per M Params'].idxmax()
fastest_idx = df_results['Acc per Min'].idxmax()

print(f"  • Highest accuracy: {df_results.loc[best_idx, 'Model']} "
      f"({df_results.loc[best_idx, 'Test Acc (%)']:.2f}%)")
print(f"  • Most parameter-efficient: {df_results.loc[most_efficient_idx, 'Model']} "
      f"({df_results.loc[most_efficient_idx, 'Acc per M Params']:.2f}% per M)")
print(f"  • Fastest training: {df_results.loc[fastest_idx, 'Model']} "
      f"({df_results.loc[fastest_idx, 'Training Time (min)']:.1f} min)")

if 'TumorNet-Lite' in df_results.iloc[0]['Model']:
    print("\n🏆 TumorNet-Lite achieves best performance!")
else:
    tumornet_rank = df_results[df_results['Model'].str.contains('TumorNet-Lite')].index[0] + 1
    print(f"\n📊 TumorNet-Lite ranks #{tumornet_rank} out of {len(df_results)} models")

print("\n" + "="*80 + "\n")

## 📝 Notes

**Fair Comparison Ensured:**
- ✅ Same preprocessed data (`preprocessed_canonical/`)
- ✅ Same training protocol (optimizer, scheduler, early stopping)
- ✅ Same random seed for reproducibility
- ✅ Same evaluation metrics (accuracy, confusion matrix, etc.)
- ✅ Same max epochs (50) with early stopping

**Next Steps:**
1. Analyze which model best fits your requirements (accuracy vs efficiency)
2. Consider ensemble methods combining top models
3. Evaluate on held-out test set for final reporting
4. Use results for paper writing and publication

---
**Experiment complete!** 🎉